# Tobit model
Pythonには、Rの `tobit()` に相当する普及・安定したライブラリが存在しない。
そのため、ここでは Python から R の AER パッケージの Tobit 関数を呼び出す方法を採用。
この方法により、左打ち切り（left=0）や右打ち切りなどの標準的なTobitモデルを安定的に推定可能。

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter
# インストールコマンド例: pip install rpy2

In [21]:
# CSVファイルの読み込み
data = pd.read_csv("../data/12-censored.csv")

In [22]:
# データを確認（先頭5件）
data.head()

,Y,Z1,Z2,Bottom
0,0.0,2.1,3.1,0.0
1,0.0,2.3,3.3,0.0
2,0.0,1.2,2.3,0.0
3,0.0,1.3,2.1,0.0
4,1.7,2.7,3.6,1.0


In [ ]:

AER = importr('AER')
ro.r('library(AER)')

In [25]:
# Python -> R データ変換（localconverterを使用）
with localconverter(ro.default_converter + pandas2ri.converter):
    r_data = ro.conversion.py2rpy(data)
ro.r.assign('data', r_data)

Y,Z1,Z2,Bottom
...,...,...,...


In [26]:
# Tobit モデル推定（左打ち切り0）
ro.r('fit <- tobit(Y ~ Z1 + Z2, left=0, data=data)')

In [27]:
# 結果表示
print(ro.r('summary(fit)'))


Call:
tobit(formula = Y ~ Z1 + Z2, left = 0, data = data)

Observations:
         Total  Left-censored     Uncensored Right-censored 
            15              4             11              0 

Coefficients:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept)  -6.5602     0.8656  -7.578  3.5e-14 ***
Z1            0.5843     0.6589   0.887 0.375203    
Z2            1.7249     0.5018   3.437 0.000588 ***
Log(scale)   -0.7417     0.2210  -3.356 0.000791 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Scale: 0.4763 

Gaussian distribution
Number of Newton-Raphson Iterations: 10 
Log-likelihood: -9.233 on 4 Df
Wald-statistic: 126.3 on 2 Df, p-value: < 2.22e-16 


